In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# NOTEBOOK OOD-SSL (ResNet-18) — PCam <-> PANDA  [PATCHED]
# - Methods: SimCLR (you can extend to BYOL later)
# - Protocol: 1 split × 3 seeds (n=3)  [additional study]
# - Label fracs: 1%, 5%, 10%
# - Fixes:
#   (1) OOD eval uses SAME img_size as ID (removes resolution confound)
#   (2) AUROC sanity-check logs flipped-score AUROC (detects sign/label issues)
#   (3) Limit figures: only seed=0 and frac=10%
# ============================================================

import os, time, random
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from PIL import Image

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss, roc_curve

import matplotlib.pyplot as plt

# -----------------------
# 0) CONFIG
# -----------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = (DEVICE.type == "cuda")
print("Device:", DEVICE)

N_SPLITS = 1
SEEDS = [0, 1, 2]
SPLIT_BASE_SEED = 1000
VAL_RATIO = 0.2
LABEL_FRACS = [0.01, 0.05, 0.10]

SSL_EPOCHS = 10
PROBE_EPOCHS = 10
LR_SSL = 1e-3
LR_PROBE = 1e-4

# debug subsampling
SAMPLE_FRAC_PCAM = 0.2
SAMPLE_FRAC_PANDA = 0.2

BATCH_SSL_RESNET = 128
BATCH_SUP_RESNET = 128

IMG_PCAM = 96
IMG_PANDA = 224

NUM_WORKERS = 4
PIN_MEMORY = True

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

RAW_CSV = f"{OUT_DIR}/results_raw_ood_ssl_resnet.csv"
SUM_CSV = f"{OUT_DIR}/results_summary_ood_ssl_resnet.csv"

# -----------------------
# 1) UTILS
# -----------------------
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def sanitize_np_probs(probs: np.ndarray) -> np.ndarray:
    probs = np.asarray(probs).reshape(-1)
    probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0)
    return np.clip(probs, 1e-7, 1 - 1e-7)

def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    probs = sanitize_np_probs(probs)
    labels = np.asarray(labels).reshape(-1).astype(int)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece, n = 0.0, len(probs)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (probs >= lo) & (probs < hi)
        if m.sum() == 0:
            continue
        ece += (m.sum()/n) * abs(labels[m].mean() - probs[m].mean())
    return float(ece)

def compute_metrics_binary(probs: np.ndarray, labels: np.ndarray, thr: float = 0.5) -> Dict[str, float]:
    probs = sanitize_np_probs(probs)
    labels = np.asarray(labels).reshape(-1).astype(int)
    preds = (probs >= thr).astype(int)
    acc = float((preds == labels).mean())
    try:
        auc = float(roc_auc_score(labels, probs))
    except ValueError:
        auc = float("nan")
    f1 = float(f1_score(labels, preds, zero_division=0))
    brier = float(brier_score_loss(labels, probs))
    ece = compute_ece(probs, labels)
    return {"acc": acc, "auc": auc, "f1": f1, "ece": ece, "brier": brier}

def append_row_csv(path: str, row: dict):
    df = pd.DataFrame([row])
    if os.path.exists(path):
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, index=False)

@torch.no_grad()
def fit_standardizer(train_feats: torch.Tensor, eps: float = 1e-6):
    mu = train_feats.float().mean(dim=0, keepdim=True)
    sd = train_feats.float().std(dim=0, keepdim=True).clamp_min(eps)
    return mu, sd

@torch.no_grad()
def apply_standardizer(feats: torch.Tensor, mu: torch.Tensor, sd: torch.Tensor):
    return (feats.float() - mu) / sd

def summarize_mean_std(raw_csv_path, group_cols, metric_cols, out_csv_path):
    df = pd.read_csv(raw_csv_path)
    g = df.groupby(group_cols)[metric_cols].agg(["mean","std","count"]).reset_index()
    g.columns = ["_".join([c for c in col if c]) for col in g.columns.values]
    g.to_csv(out_csv_path, index=False)
    return g

# -----------------------
# 2) DATA SOURCES (paths)
# -----------------------
class TwoViewSSL_Path(Dataset):
    def __init__(self, image_paths: List[str], img_size: int):
        self.image_paths = image_paths
        self.t = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
            transforms.RandomGrayscale(p=0.1),
            transforms.ToTensor(),
            transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
        ])
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return self.t(img), self.t(img)

class Supervised_Path(Dataset):
    def __init__(self, image_paths: List[str], labels: List[int], img_size: int):
        self.image_paths, self.labels = image_paths, labels
        self.t = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
        ])
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return self.t(img), int(self.labels[idx])

class PathDataSource:
    def __init__(self, name, paths, labels):
        self.name = name
        self.paths = list(paths)
        self.labels = list(map(int, labels))
        self._universe_indices = np.arange(len(self.paths), dtype=np.int64)
    def __len__(self): return len(self.paths)
    def labels_all(self): return np.asarray(self.labels, dtype=np.int64)
    def make_ssl_dataset(self, indices, img_size):
        return TwoViewSSL_Path([self.paths[i] for i in indices], img_size)
    def make_sup_dataset(self, indices, img_size):
        p = [self.paths[i] for i in indices]
        y = [self.labels[i] for i in indices]
        return Supervised_Path(p, y, img_size)

def find_any_under_input(pattern: str) -> List[Path]:
    return list(Path("/kaggle/input").rglob(pattern))

def load_pcam_datasource(sample_frac: float = 1.0) -> PathDataSource:
    base = None
    for c in find_any_under_input("histopathologic-cancer-detection/train_labels.csv"):
        base = c.parent
        break
    if base is None:
        raise FileNotFoundError("PCam missing. Add histopathologic-cancer-detection.")
    labels_path = str(base / "train_labels.csv")
    df = pd.read_csv(labels_path)
    if 0 < sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=42).reset_index(drop=True)
    train_dir = base / "train"
    df["image_path"] = df["id"].astype(str).apply(lambda x: str(train_dir / f"{x}.tif"))
    df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
    return PathDataSource("pcam", df["image_path"].tolist(), df["label"].astype(int).tolist())

def infer_panda_resized_dir() -> Optional[str]:
    candidates = find_any_under_input("train_images/train_images")
    if not candidates:
        return None
    best, best_count = None, -1
    for c in candidates:
        cnt = len(list(c.glob("*.png")))
        if cnt > best_count:
            best, best_count = c, cnt
    return str(best) if best is not None else None

def infer_panda_comp_dir() -> Optional[str]:
    candidates = find_any_under_input("train.csv")
    good = []
    for c in candidates:
        parent = c.parent
        if (parent / "sample_submission.csv").exists():
            txt = str(parent).lower()
            if "prostate" in txt or "panda" in txt:
                good.append(parent)
    if good:
        good = sorted(good, key=lambda p: len(str(p)))
        return str(good[0])
    return None

def load_panda_datasource(sample_frac: float = 1.0) -> PathDataSource:
    resized_dir = infer_panda_resized_dir()
    comp_dir = infer_panda_comp_dir()
    if resized_dir is None or comp_dir is None:
        raise FileNotFoundError("PANDA missing. Add competition + resized PNG dataset.")
    train_csv = os.path.join(comp_dir, "train.csv")
    df = pd.read_csv(train_csv)[["image_id","isup_grade"]].dropna()
    if 0 < sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=42).reset_index(drop=True)
    df["image_path"] = df["image_id"].astype(str).apply(lambda x: os.path.join(resized_dir, f"{x}.png"))
    df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
    labels = [1 if int(g) >= 2 else 0 for g in df["isup_grade"].astype(int).tolist()]
    return PathDataSource("panda", df["image_path"].tolist(), labels)

# -----------------------
# 3) SPLITS + SUBSETS
# -----------------------
def make_splits_indices(universe_indices, labels, n_splits, val_ratio, base_seed):
    labels = np.asarray(labels, dtype=np.int64)
    splits = []
    for split_id in range(n_splits):
        sss = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=base_seed + split_id)
        tr_rel, va_rel = next(sss.split(np.zeros(len(universe_indices)), labels[universe_indices]))
        splits.append((universe_indices[tr_rel], universe_indices[va_rel]))
    return splits

def stratified_label_subset_indices(train_indices, labels, frac, seed):
    labels = np.asarray(labels, dtype=np.int64)
    n = len(train_indices); k = max(1, int(n * frac))
    sss = StratifiedShuffleSplit(n_splits=1, train_size=k, random_state=seed)
    sel_rel, _ = next(sss.split(np.zeros(n), labels[train_indices]))
    return train_indices[sel_rel]

# -----------------------
# 4) RESNET-18 + SIMCLR
# -----------------------
def get_resnet18(pretrained: bool = False):
    try:
        from torchvision.models import ResNet18_Weights
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        m = models.resnet18(weights=weights)
    except Exception:
        m = models.resnet18(pretrained=pretrained)
    feat_dim = m.fc.in_features
    m.fc = nn.Identity()
    return m, feat_dim

class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, proj_dim: int = 128, hidden_dim: int = 2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, proj_dim),
        )
    def forward(self, x): return self.net(x)

def get_simclr_encoder_resnet18(proj_dim: int = 128) -> nn.Module:
    backbone, feat_dim = get_resnet18(pretrained=False)
    proj = ProjectionHead(feat_dim, proj_dim=proj_dim, hidden_dim=2048)
    return nn.Sequential(backbone, proj)

def nt_xent_loss_fp32(z1, z2, temperature=0.5):
    z1 = z1.float(); z2 = z2.float()
    b = z1.size(0)
    z1 = F.normalize(z1, dim=1); z2 = F.normalize(z2, dim=1)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(z, z.T) / float(temperature)
    mask = torch.eye(2 * b, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(mask, -1e4)
    labels = torch.arange(b, device=z.device)
    labels = torch.cat([labels + b, labels], dim=0)
    return F.cross_entropy(sim, labels)

def pretrain_simclr_resnet18(ssl_ds: Dataset, batch_size: int, epochs: int, lr: float):
    dl = DataLoader(ssl_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=PIN_MEMORY, drop_last=True)
    model = get_simclr_encoder_resnet18(proj_dim=128).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    model.train()
    for ep in range(1, epochs+1):
        total, n = 0.0, 0
        for x1, x2 in dl:
            x1 = x1.to(DEVICE, non_blocking=True)
            x2 = x2.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                z1 = model(x1)
                z2 = model(x2)
                loss = nt_xent_loss_fp32(z1, z2, temperature=0.5)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            total += float(loss.item()) * x1.size(0); n += x1.size(0)
        print(f"[SimCLR] ep {ep}/{epochs} loss={total/max(1,n):.4f}")
    return model

# -----------------------
# 5) FEATURES + PROBE
# -----------------------
def sanitize_tensor(x: torch.Tensor, clamp: float = 1e4) -> torch.Tensor:
    x = x.float()
    x = torch.nan_to_num(x, nan=0.0, posinf=clamp, neginf=-clamp)
    return torch.clamp(x, -clamp, clamp)

@torch.no_grad()
def extract_features(backbone: nn.Module, dl: DataLoader) -> Tuple[torch.Tensor, np.ndarray]:
    backbone.eval()
    feats, ys = [], []
    for x, y in dl:
        x = x.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            z = backbone(x)
        if isinstance(z, (tuple, list)): z = z[0]
        if z.ndim > 2: z = torch.flatten(z, 1)
        feats.append(sanitize_tensor(z).detach().cpu())
        ys.append(y.numpy())
    return sanitize_tensor(torch.cat(feats, 0)), np.concatenate(ys, 0).astype(int)

class LinearProbe(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Linear(d, 1)
    def forward(self, x): return self.fc(x).squeeze(1)

def train_probe(feats_std: torch.Tensor, y: np.ndarray, epochs: int, lr: float):
    X = sanitize_tensor(feats_std).float()
    y_t = torch.tensor(y, dtype=torch.float32)
    dl = DataLoader(torch.utils.data.TensorDataset(X, y_t), batch_size=256, shuffle=True)
    clf = LinearProbe(X.size(1)).to(DEVICE)
    opt = torch.optim.AdamW(clf.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss()
    clf.train()
    for ep in range(1, epochs+1):
        total, n = 0.0, 0
        for xb, yb in dl:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            loss = crit(sanitize_tensor(clf(xb)), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
            opt.step()
            total += float(loss.item()) * xb.size(0); n += xb.size(0)
        print(f"[Probe] ep {ep}/{epochs} loss={total/max(1,n):.4f}")
    return clf

@torch.no_grad()
def predict_probs(clf: nn.Module, feats_std: torch.Tensor) -> np.ndarray:
    clf.eval()
    X = sanitize_tensor(feats_std).float().to(DEVICE)
    logits = sanitize_tensor(clf(X).detach().cpu())
    return sanitize_np_probs(torch.sigmoid(logits).numpy())

# -----------------------
# 6) OOD SCORING + SANITY
# -----------------------
def ood_scores_from_probs(p: np.ndarray) -> Dict[str, np.ndarray]:
    p = sanitize_np_probs(p)
    conf = np.maximum(p, 1.0 - p)     # MSP for binary
    u = 1.0 - conf                    # uncertainty score, higher = more OOD-like
    return {"score_prob": p, "score_uncert": u}

def ood_auroc(id_scores: np.ndarray, ood_scores: np.ndarray) -> float:
    y = np.concatenate([np.zeros_like(id_scores), np.ones_like(ood_scores)])
    s = np.concatenate([id_scores, ood_scores])
    try:
        return float(roc_auc_score(y, s))
    except ValueError:
        return float("nan")

def save_hist(id_s, ood_s, path, title):
    plt.figure()
    plt.hist(id_s, bins=30, alpha=0.6, label="ID")
    plt.hist(ood_s, bins=30, alpha=0.6, label="OOD")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def save_roc(id_s, ood_s, path, title):
    y = np.concatenate([np.zeros_like(id_s), np.ones_like(ood_s)])
    s = np.concatenate([id_s, ood_s])
    fpr, tpr, _ = roc_curve(y, s)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0,1],[0,1], linestyle="--")
    plt.title(title)
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

# -----------------------
# 7) MAIN
# -----------------------
if os.path.exists(RAW_CSV):
    os.remove(RAW_CSV)

pcam = load_pcam_datasource(sample_frac=SAMPLE_FRAC_PCAM)
panda = load_panda_datasource(sample_frac=SAMPLE_FRAC_PANDA)

pairs = [("pcam","panda"), ("panda","pcam")]

def img_size_for(dsname: str) -> int:
    return IMG_PCAM if dsname == "pcam" else IMG_PANDA

for id_name, ood_name in pairs:
    id_ds = pcam if id_name == "pcam" else panda
    ood_ds = panda if ood_name == "panda" else pcam

    uni = getattr(id_ds, "_universe_indices", np.arange(len(id_ds), dtype=np.int64))
    labels_all = id_ds.labels_all()
    splits = make_splits_indices(uni, labels_all, N_SPLITS, VAL_RATIO, SPLIT_BASE_SEED)

    for split_id, (tr_idx, va_idx) in enumerate(splits):
        ood_uni = getattr(ood_ds, "_universe_indices", np.arange(len(ood_ds), dtype=np.int64))
        rng = np.random.RandomState(123)
        ood_eval_idx = rng.choice(ood_uni, size=min(2000, len(ood_uni)), replace=False)

        # ✅ FIX: use ID img_size for both ID and OOD eval to remove resolution confound
        img_id = img_size_for(id_name)

        id_val_ds = id_ds.make_sup_dataset(va_idx, img_size=img_id)
        id_val_dl = DataLoader(id_val_ds, batch_size=BATCH_SUP_RESNET, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        ood_eval_ds = ood_ds.make_sup_dataset(ood_eval_idx, img_size=img_id)  # <--- FIX
        ood_eval_dl = DataLoader(ood_eval_ds, batch_size=BATCH_SUP_RESNET, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        for seed in SEEDS:
            set_seed(10_000 + 100*split_id + seed)
            print("\n" + "="*90)
            print(f"[RUN] ID={id_name} OOD={ood_name} split={split_id} seed={seed} | img_size(ID)= {img_id}")
            print("="*90)

            # SSL pretrain on ID train set
            ssl_ds = id_ds.make_ssl_dataset(tr_idx, img_size=img_id)
            simclr = pretrain_simclr_resnet18(ssl_ds, BATCH_SSL_RESNET, SSL_EPOCHS, LR_SSL)
            backbone = simclr[0]  # encoder

            # features for ID val and OOD
            id_val_feats, id_val_y = extract_features(backbone, id_val_dl)
            ood_feats, _ = extract_features(backbone, ood_eval_dl)

            for frac in LABEL_FRACS:
                sub_tr_idx = stratified_label_subset_indices(tr_idx, labels_all, frac, seed=42 + seed + 10*split_id)
                id_tr_sup = id_ds.make_sup_dataset(sub_tr_idx, img_size=img_id)
                id_tr_dl = DataLoader(id_tr_sup, batch_size=BATCH_SUP_RESNET, shuffle=True,
                                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

                id_tr_feats, id_tr_y = extract_features(backbone, id_tr_dl)

                # standardize with TRAIN only
                mu, sd = fit_standardizer(id_tr_feats)
                id_tr_std  = apply_standardizer(id_tr_feats, mu, sd)
                id_val_std = apply_standardizer(id_val_feats, mu, sd)
                ood_std    = apply_standardizer(ood_feats, mu, sd)

                clf = train_probe(id_tr_std, id_tr_y, PROBE_EPOCHS, LR_PROBE)

                # ID metrics
                id_probs = predict_probs(clf, id_val_std)
                id_metrics = compute_metrics_binary(id_probs, id_val_y)

                # OOD
                ood_probs = predict_probs(clf, ood_std)
                id_scores = ood_scores_from_probs(id_probs)
                ood_scores = ood_scores_from_probs(ood_probs)

                # Two candidate OOD scores
                auc_prob   = ood_auroc(id_scores["score_prob"],   ood_scores["score_prob"])
                auc_uncert = ood_auroc(id_scores["score_uncert"], ood_scores["score_uncert"])

                # ✅ Sanity: flipped-score AUROC (detect sign/label issues)
                auc_prob_flip   = ood_auroc(-id_scores["score_prob"],   -ood_scores["score_prob"])
                auc_uncert_flip = ood_auroc(-id_scores["score_uncert"], -ood_scores["score_uncert"])

                print(f"[OOD] frac={int(frac*100)}% AUROC(prob)={auc_prob:.3f} flip={auc_prob_flip:.3f} | "
                      f"AUROC(uncert)={auc_uncert:.3f} flip={auc_uncert_flip:.3f}")

                row = {
                    "method": "simclr",
                    "backbone": "resnet18",
                    "id_dataset": id_name,
                    "ood_dataset": ood_name,
                    "split_id": split_id,
                    "seed": seed,
                    "label_frac": frac,
                    **id_metrics,
                    "ood_auroc_prob": auc_prob,
                    "ood_auroc_prob_flip": auc_prob_flip,
                    "ood_auroc_uncert": auc_uncert,
                    "ood_auroc_uncert_flip": auc_uncert_flip,
                    "ood_eval_n": int(len(ood_eval_idx)),
                    "img_size_id": int(img_id),
                }
                append_row_csv(RAW_CSV, row)

                # ✅ limit figures (avoid 36 png spam)
                MAKE_FIG = (seed == 0 and abs(frac - 0.10) < 1e-12)
                if MAKE_FIG:
                    tag = f"{id_name}_to_{ood_name}_s{split_id}_seed{seed}_f{int(frac*100)}"
                    save_hist(id_scores["score_uncert"], ood_scores["score_uncert"],
                              f"{OUT_DIR}/fig_ood_hist_{tag}.png",
                              f"OOD score (uncert) {tag}")
                    save_roc(id_scores["score_uncert"], ood_scores["score_uncert"],
                             f"{OUT_DIR}/fig_ood_roc_{tag}.png",
                             f"OOD ROC (uncert) {tag}")

print("\nSaved raw:", RAW_CSV)

summary = summarize_mean_std(
    RAW_CSV,
    group_cols=["method","backbone","id_dataset","ood_dataset","label_frac"],
    metric_cols=[
        "acc","auc","f1","ece","brier",
        "ood_auroc_prob","ood_auroc_prob_flip",
        "ood_auroc_uncert","ood_auroc_uncert_flip"
    ],
    out_csv_path=SUM_CSV
)
print("Saved summary:", SUM_CSV)
print(summary.head())
